In [1]:
import torch
import torch.nn as nn
import sys
import hls4ml

COMMON_PATH = "../../Common/CNet"
sys.path.append(COMMON_PATH)
try:
	from CNetPlusScalar import CNetPlusScalar
except ModuleNotFoundError as e:
	print(f"Error: Could not import MMSNet modules. Check COMMON_PATH: {COMMON_PATH}")
	raise e

In [3]:
model = CNetPlusScalar()
model.load_state_dict(torch.load(f'{COMMON_PATH}/pre_trained_w.pt', weights_only=True))

<All keys matched successfully>

In [5]:
config = hls4ml.utils.config_from_pytorch_model(
    model, 
    input_shape=[(1, 512, 512), (1,1)],
    granularity='model', 
    backend='Vitis',
    )
print("-----------------------------------")
print("Configuration")
print(config)
print("-----------------------------------")
hls_model = hls4ml.converters.convert_from_pytorch_model(
    model, hls_config=config,
    backend='Vitis',
    output_dir='model_1/hls4ml_prj',
    part='xcu250-figd2104-2L-e',
)

-----------------------------------
Configuration
{'Model': {'Precision': {'default': 'ap_fixed<16,6>'}, 'ReuseFactor': 1, 'ChannelsLastConversion': 'full', 'TransposeOutputs': False, 'Strategy': 'Latency', 'BramFactor': 1000000000, 'TraceOutput': False}, 'PytorchModel': CNetPlusScalar(
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (conv3): Conv2d(16, 32, kernel_size=(5, 5), stride=(1, 1))
  (conv4): Conv2d(32, 32, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=25088, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=85, out_features=1, bias=True)
), 'InputShape': [(1, 512, 512), (1, 1)]}
-----------------------------------
Interpreting Model ...
Topology:
Layer name: conv1, layer type: Conv2D, input shape: [[None, 1, 512, 512]]
Layer name: relu,

In [7]:
hls_model.compile()

Writing HLS project
Done


KeyboardInterrupt: 

In [8]:
hls_model.build(csim=False)

Project myproject_prj does not exist. Rerun "hls4ml build -p model_1/hls4ml_prj".


sh: vitis_hls: command not found
